# Question Answering con DistilBERT — Teoria e codice commentato

Questo notebook riprende `3_QuestionAnswering.ipynb` e lo arricchisce con teoria introduttiva, una roadmap e commenti estesi al codice.

**Obiettivo del notebook**: nei notebook precedenti (GPT-2 per il text completion, T5 per la summarization) abbiamo sempre gestito *a mano* tokenizer e modello: `tokenizer.encode(...)`, poi `model.generate(...)` o una forward pass esplicita, poi `tokenizer.decode(...)`. In questo notebook affrontiamo un task diverso — il **question answering estrattivo** — usando però uno stile di codice completamente diverso: `pipeline()`, l'API di alto livello di 🤗 Transformers, che incapsula tutti questi passaggi in un'unica chiamata. Il filo conduttore del notebook è quindi doppio:
1. capire **cos'è** il QA estrattivo e perché richiede un'architettura encoder-only come BERT/DistilBERT;
2. capire **cosa fa esattamente `pipeline()` al posto nostro**, per non trattarla come una "scatola nera" — è proprio la sezione 1.6 il cuore di questo secondo obiettivo.

---

## 1. Teoria introduttiva

### 1.1 Cos'è il Question Answering (QA) estrattivo
Il **question answering estrattivo** è il compito di individuare, all'interno di un testo di riferimento (**context**), la porzione esatta di testo (**span**) che risponde a una **domanda**. A differenza della summarization (dove il modello *genera* nuovo testo), qui il modello **seleziona** un frammento già presente nel context: non inventa parole nuove, indica solo dove inizia e dove finisce la risposta.

In pratica il modello impara a predire, per ogni token del context:
- la probabilità che quel token sia l'**inizio** della risposta;
- la probabilità che quel token sia la **fine** della risposta.

La risposta finale è la sotto-sequenza di token tra la posizione di inizio e quella di fine con probabilità (combinata) più alta.

### 1.2 BERT e l'architettura encoder-only
**BERT (Bidirectional Encoder Representations from Transformers)** usa solo lo **stack di encoder** del Transformer (a differenza di GPT-2, decoder-only, e di T5, encoder-decoder). Non essendo autoregressivo, ogni token può "guardare" **sia a sinistra che a destra** nella frase: la rappresentazione che il modello costruisce per ogni parola è quindi **bidirezionale**, il che lo rende molto efficace per compiti di *comprensione* del testo (come il QA), più che di *generazione*.

### 1.3 Pretraining di BERT
BERT è pre-addestrato con due obiettivi self-supervised (si vedano anche gli esempi nel notebook sulla summarization):
- **Masked Language Modeling (MLM)**: si maschera circa il 15% dei token in input e il modello deve predire le parole mascherate usando il contesto sia a sinistra che a destra.
- **Next Sentence Prediction (NSP)**: il modello riceve due frasi e deve predire se la seconda segue effettivamente la prima nel testo originale.

Nessuno di questi due obiettivi richiede etichette umane: sono ottenuti automaticamente a partire da testo grezzo, esattamente come il next-token prediction di GPT-2 o lo span corruption di T5.

### 1.4 DistilBERT: distillazione della conoscenza
Il modello usato in questo notebook è **DistilBERT**, una versione **compressa** di BERT ottenuta tramite **knowledge distillation**: un modello più piccolo (lo *studente*) viene addestrato a imitare il comportamento di un modello più grande già addestrato (il *maestro*, BERT). Il risultato è un modello circa il 40% più piccolo e significativamente più veloce, che mantiene comunque la maggior parte delle capacità del modello originale — un buon compromesso quando le risorse di calcolo sono limitate (nota infatti il log `Device set to use cpu`: il modello è abbastanza leggero da girare bene anche su CPU).

### 1.5 Fine-tuning per il QA
Il modello usato, `distilbert-base-cased-distilled-squad`, non è solo pre-addestrato in modo generico: è stato anche **fine-tuned** sul dataset **SQuAD** (Stanford Question Answering Dataset), che contiene migliaia di coppie (context, question, answer) etichettate da esseri umani. Questo è diverso dagli altri due notebook: lì si usava il modello pre-addestrato "puro" (GPT-2) o pre-addestrato per il task generico (T5 text-to-text); qui invece il modello ha già visto esempi supervisionati specifici per il QA estrattivo, il che spiega la qualità molto alta delle risposte anche su esempi semplici.

### 1.6 La pipeline di 🤗 Transformers

#### 1.6.1 Cos'è `pipeline()`
`pipeline()` è una **funzione factory**: non è un modello, è una funzione che *costruisce e restituisce* un oggetto `Pipeline` pronto all'uso, specializzato per un task. Internamente, la libreria mantiene un **registro dei task supportati** (una mappa `nome_task -> classe Pipeline + modello/tokenizer di default`). Quando scrivi `pipeline('question-answering', ...)`, la stringa `'question-answering'` viene cercata in questo registro: se è presente, la funzione sa quale sottoclasse Python istanziare (qui `QuestionAnsweringPipeline`) e come collegarla al modello/tokenizer che le passi.

Questo è esattamente il punto che ha causato l'errore visto in precedenza in questo notebook (`KeyError: Unknown task question-answering`): nella versione 5.x di `transformers` quel task era stato rimosso dal registro (restavano solo varianti specializzate come `document-question-answering`). Non è un bug nel nostro codice: è la libreria stessa, in una nuova major version, a non riconoscere più quella stringa. Da qui l'utilità di **fissare la versione** di `transformers` in `requirements.txt`, così l'ambiente resta riproducibile anche quando esce una nuova major version con breaking changes.

#### 1.6.2 Cosa succede quando *crei* la pipeline
La riga `pipeline('question-answering', model='distilbert-base-cased-distilled-squad', tokenizer='distilbert-base-cased-distilled-squad')` fa, in ordine:
1. **Validazione del task**: verifica che `'question-answering'` sia nel registro (v. 1.6.1).
2. **Caricamento di modello e tokenizer**: scarica (o recupera dalla cache locale) i pesi del modello e la configurazione del tokenizer dal checkpoint indicato — lo stesso `from_pretrained` visto nei notebook precedenti, solo automatizzato.
3. **Istanziazione della classe Pipeline giusta**: `QuestionAnsweringPipeline`, che *sa già* come pre-processare l'input (question + context) e come post-processare l'output (logit di inizio/fine → span di testo) per questo specifico task — logica che con GPT-2/T5 dovevamo scrivere noi a mano.

#### 1.6.3 Cosa succede quando *chiami* la pipeline
Ogni chiamata come `qa_pipeline(question=question, context=context)` esegue tre fasi, nascoste dietro un'unica riga di codice:
1. **Pre-processing**: tokenizza `question` e `context` insieme in un unico input (nel formato con cui il modello è stato fine-tuned, concettualmente `[CLS] question [SEP] context [SEP]`), gestendo anche context troppo lunghi tramite finestre scorrevoli (parametro `doc_stride`).
2. **Forward pass**: il modello produce due vettori di logit, uno per token del context — uno rappresenta "quanto è probabile che questo token sia l'*inizio*" e l'altro "quanto è probabile che sia la *fine*" della risposta.
3. **Post-processing**: la pipeline combina i logit di inizio/fine (cercando la coppia `start <= end` più probabile), converte gli indici di token in posizioni di carattere nel testo originale (usando gli offset salvati dal tokenizer) ed estrae la sottostringa corrispondente dal `context`.

Il risultato è un dizionario, ad es. `{'answer': ..., 'score': ..., 'start': ..., 'end': ...}`: `answer` è il testo estratto, `score` la confidenza (softmax) della pipeline, `start`/`end` le posizioni di carattere nel context originale.

#### 1.6.4 Alcuni parametri utili (non usati nel notebook, ma buoni da conoscere)
- `top_k`: restituisce le *k* risposte più probabili invece di una sola.
- `max_answer_len`: scarta span più lunghi di N token (evita risposte assurdamente lunghe).
- `handle_impossible_answer`: se `True`, permette alla pipeline di rispondere "nessuna risposta plausibile" quando lo score è troppo basso.
- `doc_stride` / `max_seq_len`: controllano come viene suddiviso un context più lungo della finestra di contesto del modello.

#### 1.6.5 Pipeline vs approccio manuale: cosa guadagni, cosa perdi
Nei notebook su GPT-2 e T5 il ciclo `encode → generate/forward → decode` era scritto esplicitamente, perché ogni task ha una logica di post-processing diversa (continuazione autoregressiva per GPT-2, generazione condizionata per T5). `pipeline()` offre invece la **stessa interfaccia di chiamata** — `pipeline(task, model, tokenizer)(input)` — per task concettualmente molto diversi (generazione di testo, estrazione di span, classificazione, ...), perché la logica specifica di ciascun task è incapsulata dentro la classe Pipeline corrispondente.

- **Vantaggio**: meno codice, meno probabilità di errori manuali (es. sbagliare come combinare i logit di inizio/fine), stile uniforme tra task diversi.
- **Costo**: meno controllo e visibilità sui dettagli — qui non vediamo mai esplicitamente i logit di inizio/fine, né decidiamo noi come vengono ricombinati in uno span; dobbiamo fidarci che la pipeline lo faccia correttamente per il task dichiarato.

In generale: usa l'approccio manuale (come nei notebook su GPT-2/T5) quando ti serve controllo fine o un task non standard; usa `pipeline()` per task "standard" ben supportati dalla libreria, come questo QA estrattivo.</cell_type>

## 2. Dove stiamo andando (roadmap del notebook)

Coerentemente con l'obiettivo dichiarato in apertura, la roadmap è pensata per far vedere prima *come si usa* `pipeline()` e poi *cosa produce* su esempi via via più complessi:

1. **Setup ambiente**: installiamo `tf-keras` (dipendenza di `transformers`).
2. **Import**: `torch` e la funzione `pipeline` di `transformers`.
3. **Creazione della pipeline di QA** (`pipeline('question-answering', ...)`): qui avviene tutto ciò che è descritto nella sezione 1.6.2 — validazione del task, caricamento di modello e tokenizer, istanziazione di `QuestionAnsweringPipeline`.
4. **Definizione della classe `LLMApplications`** con il metodo `question_answering`, che chiama la pipeline (sezione 1.6.3) e ne estrae la risposta testuale.
5. **Primo esempio**: una domanda semplice su una frase molto corta (definizione di "transformers").
6. **Secondo esempio**: domanda su un fatto storico (chi ha progettato la Torre Eiffel).
7. **Terzo esempio**: domanda che richiede di individuare una motivazione/scopo (perché è stata costruita la Grande Muraglia).

### 2.1 Setup dell'ambiente

In [1]:
# Installa tf-keras: dipendenza richiesta da transformers per compatibilità
# con l'API Keras legacy (il modello verrà comunque eseguito con backend PyTorch).
! pip install tf-keras

### 2.2 Import

In [2]:
# torch: framework usato sotto il cofano dalla pipeline per l'esecuzione del modello.
import torch

# pipeline: API di alto livello di transformers che incapsula
# tokenizzazione + modello + post-processing per un task standard
# (qui 'question-answering'), senza doverli gestire manualmente come
# fatto nei notebook precedenti con GPT-2 e T5.
from transformers import pipeline

c:\Users\066321758\Downloads\28-29_10 - Word embeddings e RNN\08- Modelli di pretraining_Corsini\Esercitazioni\venv-pretraining\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2.3 Creazione della pipeline di Question Answering

`distilbert-base-cased-distilled-squad` è già fine-tuned su SQuAD: la pipeline scarica sia i pesi del modello sia la configurazione del tokenizer associato, pronti per l'inferenza.

Questa è la cella che, concretamente, esegue i tre passi descritti nella sezione 1.6.2 (validazione del task → caricamento modello/tokenizer → istanziazione di `QuestionAnsweringPipeline`). È anche la cella in cui, con una versione troppo recente di `transformers`, si può incontrare l'errore `KeyError: Unknown task question-answering` discusso in 1.6.1: se capita, non è un problema del codice qui sotto, ma della versione di libreria installata (task rimosso dal registro interno).

In [ ]:
# Modello per domande e risposte (DistilBERT)
# 'question-answering': stringa-chiave cercata nel registro interno di
#   transformers (v. sezione 1.6.1). Se valida, seleziona automaticamente
#   la classe QuestionAnsweringPipeline, che sa come pre-processare
#   (tokenizzare question+context insieme) e post-processare (calcolo
#   degli span di inizio/fine risposta a partire dai logit del modello,
#   v. sezione 1.6.3) per QUESTO task specifico.
# model=...: checkpoint da cui caricare i pesi del modello.
# tokenizer=...: checkpoint da cui caricare il tokenizer. Qui coincide con
#   model perché il tokenizer è stato pubblicato assieme al modello
#   fine-tuned; passarlo esplicitamente è ridondante (pipeline lo
#   inferirebbe comunque da model) ma rende visibile che modello e
#   tokenizer devono sempre corrispondere, come già visto nei notebook
#   precedenti.
qa_pipeline = pipeline('question-answering', model='distilbert-base-cased-distilled-squad', tokenizer='distilbert-base-cased-distilled-squad')

### 2.4 La classe `LLMApplications` con il metodo `question_answering`

Stesso pattern architetturale usato nei notebook precedenti (dizionario `self.applications` estensibile), ma qui il metodo è una sottile wrapper attorno alla pipeline, che fa già quasi tutto il lavoro.

In [ ]:
class LLMApplications:
    def __init__(self):
        # Inizializza le applicazioni
        self.applications = {
            'question_answering': self.question_answering
        }

    def question_answering(self, context, question):
        """Rispondi a una domanda utilizzando DistilBERT"""

        # Un'unica chiamata alla pipeline esegue le tre fasi descritte
        # nella sezione 1.6.3 del notebook:
        # 1) pre-processing: tokenizza context e question insieme;
        # 2) forward pass: il modello produce i logit di inizio/fine
        #    risposta su ogni token del context;
        # 3) post-processing: seleziona lo span più probabile e lo
        #    ri-decodifica in testo, mappandolo alle posizioni di
        #    carattere nel context originale.
        # Il risultato è un dizionario con almeno le chiavi 'answer',
        # 'score' (confidenza) e le posizioni 'start'/'end' nel context.
        result = qa_pipeline(question=question, context=context)

        # Estrae solo il testo della risposta (ignoriamo qui score/start/end,
        # ma sono disponibili in 'result' per eventuali usi più avanzati,
        # es. per scartare risposte con score troppo basso).
        return result['answer']

### 2.5 Primo esempio: domanda su una frase molto corta

In [ ]:
llm_apps = LLMApplications()

#  Question Answering
print("QUESTION ANSWERING:")
context = "Transformers are neural networks that use attention mechanisms."
question = "What are transformers?"

# La risposta sarà uno span estratto letteralmente dal context, non una
# frase generata da zero: qui ci aspettiamo qualcosa come "neural networks".
answer = llm_apps.question_answering(context, question)
print(f"   Context: {context}")
print(f"   Question: {question}")
print(f"   Answer: {answer}")
print()

### 2.6 Secondo esempio: fatto storico (Torre Eiffel)

Domanda "who", tipicamente più semplice per un modello fine-tuned su SQuAD: la risposta è quasi sempre un'entità nominata (nome proprio) chiaramente identificabile nel testo.

In [ ]:
# Another Question Answering Example
print("QUESTION ANSWERING:")
context = "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It was named after the engineer Gustave Eiffel, whose company designed and built the tower."
question = "Who designed the Eiffel Tower?"
answer = llm_apps.question_answering(context, question)
print(f"   Context: {context}")
print(f"   Question: {question}")
print(f"   Answer: {answer}")
print()

### 2.7 Terzo esempio: domanda sullo scopo/motivazione (Grande Muraglia)

Domanda "what was the purpose", più impegnativa di una semplice "who": la risposta non è un'entità nominata isolata, ma una porzione di frase che esprime uno scopo ("to protect northern China"). Utile per osservare che il modello sa individuare span più lunghi e semanticamente più complessi, non solo singole parole.

In [ ]:
# Another Question Answering Example
print("QUESTION ANSWERING:")
context = "The Great Wall of China is a series of fortifications made of various materials like stone, brick, tamped earth, and others. It was built to protect northern China from invasions and raids by various nomadic groups of the Eurasian Steppe."
question = "What was the purpose of the Great Wall of China?"
answer = llm_apps.question_answering(context, question)
print(f"   Context: {context}")
print(f"   Question: {question}")
print(f"   Answer: {answer}")
print()

## 3. Cosa osservare nei risultati

- Le risposte sono sempre **estratte letteralmente** dal context: il modello non può "inventare" parole che non siano già presenti nel testo di input. Se la risposta corretta non è contenuta nel context, il modello restituirà comunque lo span che considera più probabile, ma sarà per forza scorretto o parziale.
- A differenza dei due notebook precedenti (GPT-2 e T5), qui non c'è generazione libera: il compito è **selezione**, non produzione di nuovo testo. Per questo il risultato è deterministico (nessun `do_sample`), a parità di context/question.
- La qualità molto alta delle risposte, anche su esempi semplici, è dovuta al **fine-tuning su SQuAD**: il modello ha già visto migliaia di esempi di coppie (context, question, answer) etichettate da persone, non solo il pretraining self-supervised generico (MLM + NSP).
- Il log `Device set to use cpu` conferma che, grazie alla distillazione, il modello è abbastanza leggero da essere eseguito comodamente anche senza GPU.
- Il dizionario `result` restituito dalla pipeline contiene anche `score` (confidenza della risposta) e le posizioni `start`/`end` nel context: in un'applicazione reale sarebbe buona pratica controllare `score` per scartare risposte poco affidabili, invece di restituire sempre e comunque `result['answer']`.